# Testing notebook to train a VAE to compress BEV features in a lossless way

https://huggingface.co/stabilityai/sd-vae-ft-mse#:~:text=from%20diffusers.models%20import%20AutoencoderKL,Decoder%20Finetuning

In [1]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from tqdm import tqdm
import os
import time
from data.dataset import BEVFeaturesDataset, PaddDataset
from torch.utils.data import DataLoader, Dataset
import wandb

from torchinfo import summary
os.environ['CUDA_LAUNCH_BLOCKING'] = "1"

In [2]:
print(torch.__version__)

1.10.0


In [3]:
from diffusers import AutoencoderKL

In [4]:
device = 'cuda:3'
model_config = dict(
    in_channel=32,
    out_channel=32,
    inner_channel=128,
    norm_groups=32,
    channel_mults=(1, 2, 4, 8),
    attn_res=(25,),
    res_blocks=2,
    dropout=0,
)

training_config = dict(
    num_epochs=100,
    learning_rate=1e-4,
    weight_decay=1e-4,
    val_interval=1,
    save_interval=10,
    batch_size=4)

In [5]:
vae = AutoencoderKL(in_channels=256, 
                    out_channels=256, 
                    latent_channels=32,
                    block_out_channels=(256, 256, 512, 512),
                    down_block_types=("DownEncoderBlock2D", "DownEncoderBlock2D", "DownEncoderBlock2D", "AttnDownEncoderBlock2D"),
                    up_block_types=("UpDecoderBlock2D", "UpDecoderBlock2D", "UpDecoderBlock2D", "AttnUpDecoderBlock2D"),
                    layers_per_block=2,
                    ).to(device)

In [1]:
batch_size =1
summary(vae, input_size=(batch_size, 256, 200, 200))

NameError: name 'summary' is not defined

In [ ]:
vae.eval()
with torch.no_grad():
    recon = vae.forward(torch.randn(1, 256, 200, 200).to(device), sample_posterior=False, return_dict=True)

In [ ]:
print(recon)

DecoderOutput(sample=tensor([[[[-0.0070,  0.1298,  0.0772,  ...,  0.1773,  0.1264,  0.0737],
          [ 0.0318,  0.1793,  0.0746,  ...,  0.1380,  0.1014,  0.0535],
          [-0.0034,  0.1768,  0.0753,  ...,  0.2150,  0.1499,  0.0576],
          ...,
          [-0.0210,  0.1026,  0.0431,  ...,  0.2316,  0.2528,  0.1373],
          [-0.0226,  0.1103,  0.0851,  ...,  0.2136,  0.2150,  0.1375],
          [-0.0056,  0.0465,  0.0393,  ...,  0.1804,  0.1461,  0.0780]],

         [[ 0.0167,  0.1280,  0.0869,  ...,  0.2268,  0.2068,  0.2661],
          [ 0.1056,  0.2647,  0.2440,  ...,  0.3350,  0.2735,  0.2463],
          [ 0.1713,  0.3357,  0.2916,  ...,  0.4256,  0.3221,  0.2603],
          ...,
          [ 0.1226,  0.2462,  0.2104,  ...,  0.1664,  0.1725,  0.2168],
          [ 0.1737,  0.2286,  0.2238,  ...,  0.1674,  0.1848,  0.2034],
          [ 0.1473,  0.1841,  0.1721,  ...,  0.1712,  0.1183,  0.0272]],

         [[ 0.1782,  0.1592,  0.1587,  ...,  0.1805,  0.0756,  0.0220],
         

In [ ]:
vae.eval()
out = vae.encode(torch.randn(batch_size, 256, 200, 200).to(device), return_dict=True)
print(out.latent_dist.sample().shape)

torch.Size([1, 32, 25, 25])


In [6]:
class RollingStatistics(nn.Module):
    """
    Class for capturing the rolling statistics when doing online training. Calculates the mean and variance of the data per batch and updates the overall mean and variance using https://notmatthancock.github.io/2017/03/23/simple-batch-stat-updates.html formula.

    For online training only do it for one epoch after we have the overall statistics, then we can fix the statistics and do normalization and denormalization using the fixed statistics. This is because the statistics will not change much after one epoch of training, and it will be more stable to use fixed statistics for normalization and denormalization during training.

    Attributes:
        mean: The rolling mean of the data.
        var: The rolling variance of the data.
        std: The rolling standard deviation of the data.
        p_samples: The total number of observations seen so far. Used for calculating the new mean and variance when a new batch of data is observed. p_samples is the amount of samples in a batch.
    """
    def __init__(self, p_samples=0, channels=256):
        super().__init__()
        self.fixed = False
        # self.mean = mean
        # self.var = var
        # self.std = std
        self.p_samples = p_samples
        self.channels = channels

        # register_buffer ensures these move with the model and save in state_dict
        self.register_buffer('mean', torch.zeros(1, channels, 1, 1))
        self.register_buffer('var', torch.ones(1, channels, 1, 1))
        self.register_buffer('std', torch.ones(1, channels, 1, 1))
        # self.register_buffer('p_samples', torch.tensor(0, dtype=torch.long))

    # def set_device(self, device):
    #     if self.mean is not None:
    #         self.mean = self.mean.to(device)
    #     if self.var is not None:
    #         self.var = self.var.to(device)
    #     if self.std is not None:
    #         self.std = self.std.to(device)

    def __str__(self):
        return f"RollingStatistics(mean={torch.mean(self.mean)}, var={torch.mean(self.var)}, std={torch.mean(self.std)}, p_samples={self.p_samples})"

    def update(self, batch_data):
        """
        batch_data shape: (B, C, H, W)
        """
        if self.fixed:
            return

        # Calculate current batch stats
        dims = (0, 2, 3)
        batch_mean = torch.mean(batch_data, dim=dims, keepdim=True)
        batch_var = torch.var(batch_data, dim=dims, keepdim=True, correction=0)
        
        # Calculate q_samples based on ALL elements reduced (B * H * W)
        q_samples = batch_data.shape[0] * batch_data.shape[2] * batch_data.shape[3]

        if self.p_samples == 0:
            self.mean.copy_(batch_mean)
            self.var.copy_(batch_var)
            self.p_samples = q_samples
        else:
            n_p = self.p_samples
            n_q = q_samples
            n_total = n_p + n_q

            # Welford's Parallel Mean
            new_mean = (n_p * self.mean + n_q * batch_mean) / n_total

            # Welford's Parallel Variance
            # Formula: [(n_p * var_p + n_q * var_q) / n_total] + [n_p * n_q * (mean_p - mean_q)^2 / n_total^2]
            mean_diff = self.mean - batch_mean
            new_var = ((n_p * self.var + n_q * batch_var) / n_total) + \
                      ((n_p * n_q) * torch.pow(mean_diff, 2) / (n_total ** 2))

            self.mean.copy_(new_mean)
            self.var.copy_(new_var)
            self.p_samples = n_total

        self.std.copy_(torch.sqrt(self.var + 1e-8))

    def normalize(self, sample):
        return (sample - self.mean) / self.std

    def denormalize(self, sample):
        return (sample * self.std) + self.mean

    def get_stats(self):
        return {'mean': self.mean, 'var': self.var, 'std': self.std, 'p_samples': self.p_samples, 'channels': self.channels}
    

In [7]:
rolling_stats_img = RollingStatistics(channels=256)
rolling_stats_pts = RollingStatistics(channels=256)

In [8]:
from torch.utils.data import DataLoader, Subset

def load_data():
    # Load the saved data
    dataset = BEVFeaturesDataset(root_dir='/home/mingdayang/FeatureBridgeMapping/data/bev_features', transform=None)

    return dataset

def create_splits(dataset, train_split=0.8):
    gen = torch.Generator()
    gen.manual_seed(0)
    train_dataset, test_dataset = torch.utils.data.random_split(dataset, [int(train_split * len(dataset)), len(dataset) - int(train_split * len(dataset))], generator=gen)

    return train_dataset, test_dataset

def make_loader(batch_size, dataset):

    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)
        # Let's check out what we've created

    return dataloader

dataset = load_data()
dataset_10_samples = Subset(dataset, [0, 1, 2, 3, 4, 5, 6, 7, 8, 9])
train_dataset, test_dataset = create_splits(dataset)
train_loader = make_loader(training_config['batch_size'], train_dataset)
test_loader = make_loader(training_config['batch_size'], test_dataset)
single_loader = DataLoader(Subset(dataset, [0]), batch_size=training_config['batch_size'], shuffle=False)
print(len(train_dataset), len(test_dataset))


64 17


In [ ]:
# simple training loop using the existing `tensor` as toy data
device = 'cuda:3'
vae.to(device)
vae.train()
rolling_stats_img.to(device)
rolling_stats_pts.to(device)

optimizer = torch.optim.Adam(vae.parameters(), lr=1e-4)

epochs = 100
val_interval = training_config['val_interval']

try:
    with wandb.init(project="vae_test", config=model_config) as run:
        run.watch(vae, log="all", log_freq=10)
        for epoch in range(epochs):
            train_loss = 0
            for batch in tqdm(train_loader, desc=f'Epoch {epoch+1}/{epochs}'):
                X, y = batch['img_bev_embed'], batch['pts_bev_embed']
                X = X.to(device)
                if epoch > 1:
                    rolling_stats_img.fixed = True
                
                rolling_stats_img.update(X)

                X = rolling_stats_img.normalize(X)

                print(f"Rolling stats img: {rolling_stats_img}")
                
                print(f"X min max: {torch.amin(X)}, {torch.amax(X)}")
                
                posterior = vae.tiled_encode(X).latent_dist
                z = posterior.sample()
                reconstructions = vae.tiled_decode(z).sample

                mean = posterior.mean
                logvar = posterior.logvar

                kl_loss = posterior.kl().mean()
                recon_loss = F.mse_loss(reconstructions, X, reduction='mean')
                loss = recon_loss + 1e-5*kl_loss


                nn.utils.clip_grad_norm_(vae.parameters(), max_norm=5.0)
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
                train_loss += loss.item()
                print(f"Batch Loss: {loss.item()}")
            
            avg_train_loss = train_loss / len(train_loader)
            run.log({'train_loss': avg_train_loss}, step=epoch)
            if avg_train_loss < 0.01:
                print(f"Early stopping at epoch {epoch+1} with average training loss {avg_train_loss}")
                break

            if (epoch + 1) % val_interval == 0:
                with torch.no_grad():
                    val_loss = 0
                    for batch in tqdm(test_loader, desc=f'Validation Epoch {epoch+1}'):
                        X, y = batch['img_bev_embed'], batch['pts_bev_embed']
                        X = X.to(device)
                        if epoch > 1:
                            rolling_stats_img.fixed = True
                        
                        X = rolling_stats_img.normalize(X)

                        posterior = vae.tiled_encode(X).latent_dist
                        z = posterior.sample()
                        reconstructions = vae.tiled_decode(z).sample

                        mean = posterior.mean
                        logvar = posterior.logvar

                        kl_loss = posterior.kl().mean()
                        recon_loss = F.mse_loss(reconstructions, X, reduction='mean')
                        loss = recon_loss + 1e-5*kl_loss
                        val_loss += loss.item()
                        print(f"Validation Batch Loss: {loss.item()}")
                    
                    avg_val_loss = val_loss / len(test_loader)
                    run.log({'val_loss': avg_val_loss}, step=epoch)

        run.finish()
except (BrokenPipeError, KeyboardInterrupt) as e:
    print(f"Training interrupted: {e}")
    wandb.finish()
finally:
    wandb.finish()


wandb: Using wandb-core as the SDK backend.  Please refer to https://wandb.me/wandb-core for more information.
wandb: Currently logged in as: mln9d4 (mln9d4-tu-delft). Use `wandb login --relogin` to force relogin


Epoch 1/100:   0%|          | 0/16 [00:00<?, ?it/s]

Rolling stats img: RollingStatistics(mean=-0.009384727105498314, var=0.04068787023425102, std=0.16667628288269043, p_samples=160000)
X min max: -10.907147407531738, 9.617388725280762


Epoch 1/100:   6%|▋         | 1/16 [00:12<03:01, 12.13s/it]

Batch Loss: 1.1703670024871826
Rolling stats img: RollingStatistics(mean=-0.009148651733994484, var=0.04097837209701538, std=0.16694563627243042, p_samples=320000)
X min max: -10.759415626525879, 9.953228950500488


...

Batch Loss: 0.9614053964614868
Rolling stats img: RollingStatistics(mean=-0.009991242550313473, var=0.041641056537628174, std=0.1680113673210144, p_samples=5120000)
X min max: -11.069740295410156, 10.38308334350586


Epoch 5/100:  25%|██▌       | 4/16 [00:40<02:03, 10.29s/it]

Batch Loss: 0.9546306729316711
Rolling stats img: RollingStatistics(mean=-0.009991242550313473, var=0.041641056537628174, std=0.1680113673210144, p_samples=5120000)
X min max: -11.263561248779297, 9.973307609558105


Epoch 5/100:  31%|███▏      | 5/16 [00:50<01:52, 10.23s/it]

Batch Loss: 0.8955409526824951
Rolling stats img: RollingStatistics(mean=-0.009991242550313473, var=0.041641056537628174, std=0.1680113673210144, p_samples=5120000)
X min max: -11.037352561950684, 10.704817771911621
